# Best Model retraining 

Best performing model for motor control retrained using data from all motors


In [4]:
# some_file.py
import sys
sys.path.insert(1, '../')

import nn_fncs
import numpy as np
# Read 8 motors control data from .mat files and stack into input and output arrays
for i in range(1, 9):
    mat_data = nn_fncs.read_mat_workspace(f'./NewData/motorModelControl{i}.mat') # data sampled at 1e-4s, only 100 trajectories
    motor_p = mat_data.get('motor_p_out')
    if i == 1:
        input_data = motor_p[:,:, 1:4]
        output_data = motor_p[:,:, 0]
    else:
        input_data = np.vstack((input_data, motor_p[:,:, 1:4]))
        output_data = np.vstack((output_data, motor_p[:,:, 0]))

# print input and output data shapes
print(f'input_data shape: {input_data.shape}')  # (num_trajectories, num_time_steps, PID errors)
print(f'output_data shape: {output_data.shape}')  # (num_trajectories, num_time_steps, u signals)

c:\Users\brend\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\io\matlab\_mio.py:235: MatReadWarning: Duplicate variable name "None" in stream - replacing previous with new
Considerscipy.io.matlab.varmats_from_mat to split file into single variable files
  matfile_dict = MR.get_variables(variable_names)


input_data shape: (4000, 5000, 3)
output_data shape: (4000, 5000)


In [5]:
# Split data into training and validation sets
import numpy as np
train_ratio = 0.8
train_size = int(input_data.shape[0] * train_ratio)

train_input = input_data[:train_size].reshape(-1, input_data.shape[2])
train_output = output_data[:train_size].reshape(-1, 1)
val_input = input_data[train_size:].reshape(-1, input_data.shape[2])
val_output = output_data[train_size:].reshape(-1, 1)

# Max Abs values before scaling
print(f'Max abs input before scaling: {np.max(np.abs(train_input), axis=0)}')
print(f'Max abs output before scaling: {np.max(np.abs(train_output), axis=0)}')

# # Standardize the data
# Load scalers if they exist, else create new ones
from joblib import load

input_scaler = load('./ControlScalers/input_scaler.pkl')
output_scaler = load('./ControlScalers/output_scaler.pkl')
print("Scalers loaded successfully.")

train_input = input_scaler.transform(train_input)
train_output = output_scaler.transform(train_output)
val_input = input_scaler.transform(val_input)
val_output = output_scaler.transform(val_output)

# Print max abs values after scaling
print(f'Max abs input after scaling: {np.max(np.abs(train_input), axis=0)}')
print(f'Max abs output after scaling: {np.max(np.abs(train_output), axis=0)}')

# Create datasets and dataloaders
import torch
from torch.utils.data import TensorDataset, DataLoader
train_dataset = TensorDataset(torch.tensor(train_input, dtype=torch.float32),
                              torch.tensor(train_output, dtype=torch.float32))
val_dataset = TensorDataset(torch.tensor(val_input, dtype=torch.float32),
                            torch.tensor(val_output, dtype=torch.float32))

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

Max abs input before scaling: [176728.38890631    515.95320311    515.95320311]
Max abs output before scaling: [12.]
Scalers loaded successfully.
Max abs input after scaling: [1989.87808704  244.48800819  244.48800819]
Max abs output after scaling: [7.60145786]


In [6]:
# MODEL STRUCTURE

import torch
import torch.nn as nn
import torch.nn.functional as F

n_neurons = 32
class ControlModelv2(nn.Module):
    def __init__(self, input_size=3, output_size=1, n_neurons=n_neurons):  
        super(ControlModelv2, self).__init__()
        # Fully connected layers
        self.fc1 = nn.Linear(input_size, n_neurons) 
        self.fc2 = nn.Linear(n_neurons, n_neurons)
        self.output = nn.Linear(n_neurons, output_size)
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, errors):
        dx = F.relu(self.fc1(errors))
        dx = self.dropout(dx)
        dx = F.relu(self.fc2(dx))
        dx = self.output(dx)
        return dx

# Create the model
model = ControlModelv2()

# Print model summary
print(model)
print(f'Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}')

ControlModelv2(
  (fc1): Linear(in_features=3, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=32, bias=True)
  (output): Linear(in_features=32, out_features=1, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)
Total parameters: 1217


In [7]:
# Load current triained model weights
import torch    
model.load_state_dict(torch.load("control_model_v2_retrained2.pth"))

<All keys matched successfully>

In [8]:
# One step validation
from torcheval.metrics.functional import mean_squared_error, r2_score
from scipy.integrate import odeint

def one_step_validation(model, criterion, input_data, target_data):
    model.eval()
    with torch.no_grad():
        # Forward pass
        pred = model(input_data)  # Integrate over a small time step
        loss = criterion(pred, target_data)  # Compare with the next state
    r2 = r2_score(pred, target_data)
    return pred, loss, r2

In [9]:
# One step training
from torcheval.metrics.functional import mean_squared_error, r2_score
from scipy.integrate import odeint

def one_step_training(model, criterion, optimizer, input_data, target_data):
    model.train()
    with torch.set_grad_enabled(True):
        # Forward pass
        pred = model(input_data)  # Integrate over a small time step
        loss = criterion(pred, target_data)  # Compare with the next state
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    r2 = r2_score(pred, target_data)
    return pred, loss, r2

In [10]:
# Complete training loop
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

n_epochs = 100
epoch_update = 10
with tqdm(total=n_epochs) as pbar:
    for epoch in range(n_epochs):
        model.train()
        train_losses = []
        train_r2s = []
        for input_data, target_data in train_loader:
            pred, loss, r2 = one_step_training(model, criterion, optimizer, input_data, target_data)
            train_losses.append(loss.item())
            train_r2s.append(r2.item())
        
        model.eval()
        val_losses = []
        val_r2s = []
        with torch.no_grad():
            for input_data, target_data in val_loader:
                pred, loss, r2 = one_step_validation(model, criterion, input_data, target_data)
                val_losses.append(loss.item())
                val_r2s.append(r2.item())
        
        if (epoch + 1) % epoch_update == 0:
            print(f'Epoch {epoch+1}/{n_epochs}, '
                f'Train Loss: {sum(train_losses)/len(train_losses):.4f}, '
                f'Train R2: {sum(train_r2s)/len(train_r2s):.4f}, '
                f'Val Loss: {sum(val_losses)/len(val_losses):.4f}, '
                f'Val R2: {sum(val_r2s)/len(val_r2s):.4f}')
            pbar.update(epoch_update)

 10%|█         | 10/100 [1:01:34<9:14:10, 369.45s/it]

Epoch 10/100, Train Loss: 4.4426, Train R2: 0.8481, Val Loss: 4.5269, Val R2: -inf


 20%|██        | 20/100 [2:04:05<8:17:01, 372.77s/it]

Epoch 20/100, Train Loss: 4.4310, Train R2: 0.8485, Val Loss: 4.4263, Val R2: -inf


 30%|███       | 30/100 [3:06:22<7:15:24, 373.21s/it]

...

Epoch 80/100, Train Loss: 4.4149, Train R2: 0.8490, Val Loss: 4.4812, Val R2: -inf


 90%|█████████ | 90/100 [9:20:48<1:02:19, 373.91s/it]

Epoch 90/100, Train Loss: 4.4161, Train R2: 0.8490, Val Loss: 4.4636, Val R2: -inf


100%|██████████| 100/100 [10:22:46<00:00, 373.67s/it]

Epoch 100/100, Train Loss: 4.4170, Train R2: 0.8489, Val Loss: 4.5106, Val R2: -inf


In [11]:
save_path = "control_model_v2_retrained4.pth"
torch.save(model.state_dict(), save_path)
print(f'Model saved to {save_path}')

Model saved to control_model_v2_retrained4.pth
